In [ ]:
# Ячейка 1: Импорты и создание окон
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

# Создаем окна данных (симуляция времени)
np.random.seed(42)
n = len(cardio)

# Добавляем сдвиги в данных
scenarios = {
    'reference': {'start': 0, 'end': 200, 'shift': 0},
    'covariate_drift': {'start': 200, 'end': 350, 'shift': 1},
    'prior_drift': {'start': 350, 'end': 500, 'shift': 2},
}

# Применяем сдвиги
for ds_name, df in [('cardio', cardio), ('credit', credit)]:
    df['window_id'] = 'reference'
    df.loc[200:350, 'window_id'] = 'covariate_drift'
    df.loc[350:, 'window_id'] = 'prior_drift'
    
    # Симуляция сдвига: меняем распределение thalach/credit_history
    if ds_name == 'cardio':
        df.loc[200:350, 'thalach'] = df.loc[200:350, 'thalach'] * 0.8
        df.loc[350:, 'target'] = np.random.binomial(1, 0.6, 150)
    else:
        df.loc[200:350, 'credit_history'] = np.random.randint(0, 3, 150)
        df.loc[350:, 'default'] = np.random.binomial(1, 0.5, 150)

print("Окна созданы:")
print(cardio['window_id'].value_counts())

In [ ]:
# Ячейка 2: Детекция дрифта
features_cardio = ['thalach', 'oldpeak', 'age']
features_credit = ['credit_history', 'debt_to_income', 'age']

drift_results = []

for ds_name, df, features in [
    ('cardiovascular_risk', cardio, features_cardio),
    ('credit_risk', credit, features_credit)
]:
    ref_data = df[df['window_id'] == 'reference']
    
    for window in ['covariate_drift', 'prior_drift']:
        cur_data = df[df['window_id'] == window]
        
        for feature in features:
            ref_vals = ref_data[feature].values
            cur_vals = cur_data[feature].values
            
            feature_type = 'numerical' if feature != 'loan_purpose' else 'categorical'
            
            if feature_type == 'numerical':
                stat, p_value = lab_utils.ks_test(ref_vals, cur_vals)
                detector = 'KS'
            else:
                stat, p_value = lab_utils.chi2_test(ref_vals, cur_vals)
                detector = 'chi2'
            
            psi = lab_utils.calculate_psi(ref_vals, cur_vals)
            
            drift_flag = (p_value < lab_utils.ALPHA) or (psi > lab_utils.PSI_WARN)
            
            drift_results.append({
                'dataset': ds_name,
                'window_id': window,
                'scenario': window,
                'feature': feature,
                'feature_type': feature_type,
                'detector': detector,
                'statistic': stat,
                'p_value': p_value,
                'effect_size': psi,
                'drift_flag': drift_flag
            })

drift_df = pd.DataFrame(drift_results)
drift_df.to_csv('outputs/drift_detection_audit.csv', index=False)
print("Обнаружен дрифт:")
print(drift_df[drift_df['drift_flag']][['dataset','window_id','feature','p_value','effect_size']])

In [ ]:
# Ячейка 3: Мониторинг качества
quality_results = []

for ds_name, df, features, target in [
    ('cardiovascular_risk', cardio, features_cardio, 'target'),
    ('credit_risk', credit, features_credit, 'default')
]:
    # Reference модель
    ref_data = df[df['window_id'] == 'reference']
    X_ref = ref_data[features]
    y_ref = ref_data[target]
    
    scaler = StandardScaler()
    X_ref_s = scaler.fit_transform(X_ref)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_ref_s, y_ref)
    
    # Эталонные метрики
    y_ref_pred = model.predict(X_ref_s)
    y_ref_prob = model.predict_proba(X_ref_s)[:, 1]
    
    ref_f1 = f1_score(y_ref, y_ref_pred)
    ref_cost = lab_utils.expected_cost(y_ref.values, y_ref_pred)
    
    # Проверяем на всех окнах
    for window in df['window_id'].unique():
        cur_data = df[df['window_id'] == window]
        X_cur = cur_data[features]
        y_cur = cur_data[target]
        
        X_cur_s = scaler.transform(X_cur)
        y_cur_pred = model.predict(X_cur_s)
        y_cur_prob = model.predict_proba(X_cur_s)[:, 1]
        
        cur_f1 = f1_score(y_cur, y_cur_pred, zero_division=0)
        cur_cost = lab_utils.expected_cost(y_cur.values, y_cur_pred)
        
        quality_results.append({
            'dataset': ds_name,
            'window_id': window,
            'scenario': window,
            'model_variant': 'LogisticRegression',
            'accuracy': (y_cur_pred == y_cur).mean(),
            'f1': cur_f1,
            'roc_auc': roc_auc_score(y_cur, y_cur_prob),
            'pr_auc': 0,
            'brier': brier_score_loss(y_cur, y_cur_prob),
            'ece': 0,
            'expected_cost': cur_cost,
            'delta_f1_vs_reference': cur_f1 - ref_f1,
            'delta_cost_vs_reference': cur_cost - ref_cost
        })

quality_df = pd.DataFrame(quality_results)
quality_df.to_csv('outputs/monitoring_quality_audit.csv', index=False)
print("Мониторинг качества:")
print(quality_df[['dataset','window_id','f1','expected_cost','delta_f1_vs_reference']])